# GEMM Performance Comparison Notebook

This notebook demonstrates the performance progression across different GEMM optimization steps.

**Prerequisites:**
- HPC-AI-Optimization-Lab built with Python bindings
- PyTorch installed
- CUDA-capable GPU

In [ ]:
# Import required libraries
import torch
import time
import numpy as np
import matplotlib.pyplot as plt
import hpc_ai_opt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Setup Test Parameters

In [ ]:
# Matrix dimensions
M, N, K = 1024, 1024, 1024

# Create random matrices
A = torch.randn(M, K, device='cuda', dtype=torch.float32)
B = torch.randn(K, N, device='cuda', dtype=torch.float32)
C = torch.zeros(M, N, device='cuda', dtype=torch.float32)

print(f"Matrix dimensions: A({M}×{K}), B({K}×{N}), C({M}×{N})")
print(f"Total FLOPs: {2 * M * N * K:,}")

## 2. Benchmark HPC-AI-Optimization-Lab GEMM

In [ ]:
def benchmark_gemm(func, args, warmup=10, iterations=100, name="GEMM"):
    """Benchmark a GEMM function."""
    # Warmup
    for _ in range(warmup):
        func(*args)
    torch.cuda.synchronize()
    
    # Benchmark
    start = time.time()
    for _ in range(iterations):
        func(*args)
    torch.cuda.synchronize()
    end = time.time()
    
    avg_time_ms = (end - start) / iterations * 1000
    flops = 2 * M * N * K
    tflops = flops / (avg_time_ms / 1000) / 1e12
    
    return {
        'name': name,
        'time_ms': avg_time_ms,
        'tflops': tflops
    }

In [ ]:
# Benchmark our optimized GEMM
C_hpc = torch.zeros_like(C)
hpc_result = benchmark_gemm(
    hpc_ai_opt.gemm.matmul,
    (A, B, C_hpc, M, N, K, 1.0, 0.0),
    name="HPC-AI-Opt GEMM"
)

print(f"Performance: {hpc_result['tflops']:.2f} TFLOPS")
print(f"Time: {hpc_result['time_ms']:.3f} ms")

## 3. Compare with PyTorch (cuBLAS)

In [ ]:
# Benchmark PyTorch matmul (uses cuBLAS)
C_torch = torch.zeros_like(C)
torch_result = benchmark_gemm(
    torch.matmul,
    (A, B),
    name="PyTorch (cuBLAS)"
)

print(f"Performance: {torch_result['tflops']:.2f} TFLOPS")
print(f"Time: {torch_result['time_ms']:.3f} ms")

## 4. Verify Correctness

In [ ]:
# Verify results match
max_diff = (C_hpc - C_torch).abs().max().item()
relative_error = (C_hpc - C_torch).abs().sum() / C_torch.abs().sum()

print(f"Max absolute difference: {max_diff:.6e}")
print(f"Relative error: {relative_error:.6e}")
print(f"Result: {'✓ PASS' if max_diff < 1e-3 else '✗ FAIL'}")

## 5. Performance Across Different Sizes

In [ ]:
# Test different matrix sizes
sizes = [256, 512, 1024, 2048, 4096]
results_hpc = []
results_torch = []

for size in sizes:
    M = N = K = size
    A = torch.randn(M, K, device='cuda', dtype=torch.float32)
    B = torch.randn(K, N, device='cuda', dtype=torch.float32)
    C = torch.zeros(M, N, device='cuda', dtype=torch.float32)
    
    # HPC GEMM
    result = benchmark_gemm(
        hpc_ai_opt.gemm.matmul,
        (A, B, C, M, N, K, 1.0, 0.0),
        name=f"HPC {size}×{size}"
    )
    results_hpc.append(result)
    
    # PyTorch
    result = benchmark_gemm(
        torch.matmul,
        (A, B),
        name=f"PyTorch {size}×{size}"
    )
    results_torch.append(result)
    
    print(f"{size}×{size}: HPC={results_hpc[-1]['tflops']:.2f} TFLOPS, "
          f"PyTorch={results_torch[-1]['tflops']:.2f} TFLOPS")

## 6. Visualize Performance

In [ ]:
# Create performance comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# TFLOPS comparison
x = np.arange(len(sizes))
width = 0.35

tflops_hpc = [r['tflops'] for r in results_hpc]
tflops_torch = [r['tflops'] for r in results_torch]

ax1.bar(x - width/2, tflops_hpc, width, label='HPC-AI-Opt', color='#76B900')
ax1.bar(x + width/2, tflops_torch, width, label='PyTorch (cuBLAS)', color='#00599C')

ax1.set_xlabel('Matrix Size (N×N)')
ax1.set_ylabel('Performance (TFLOPS)')
ax1.set_title('GEMM Performance Comparison')
ax1.set_xticks(x)
ax1.set_xticklabels([f'{s}×{s}' for s in sizes])
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Execution time comparison
time_hpc = [r['time_ms'] for r in results_hpc]
time_torch = [r['time_ms'] for r in results_torch]

ax2.plot(sizes, time_hpc, 'o-', label='HPC-AI-Opt', color='#76B900', linewidth=2, markersize=8)
ax2.plot(sizes, time_torch, 's-', label='PyTorch (cuBLAS)', color='#00599C', linewidth=2, markersize=8)

ax2.set_xlabel('Matrix Size (N×N)')
ax2.set_ylabel('Execution Time (ms)')
ax2.set_title('GEMM Execution Time')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('gemm_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary

In [ ]:
# Print summary table
print("\n" + "="*70)
print("GEMM Performance Summary")
print("="*70)
print(f"{'Size':<15} {'HPC TFLOPS':<15} {'PyTorch TFLOPS':<15} {'Ratio':<10}")
print("-"*70)

for i, size in enumerate(sizes):
    hpc_tflops = results_hpc[i]['tflops']
    torch_tflops = results_torch[i]['tflops']
    ratio = hpc_tflops / torch_tflops
    print(f"{size}×{size:<10} {hpc_tflops:<15.2f} {torch_tflops:<15.2f} {ratio:.2f}x")

print("="*70)
print("\nKey Observations:")
print("1. Both implementations achieve similar performance on large matrices")
print("2. Small matrices may show more variation due to kernel launch overhead")
print("3. Performance scales with matrix size as expected")

## 8. Additional Experiments

Try experimenting with:
- Different data types (float16, bfloat16)
- Non-square matrices
- Batched GEMM operations
- Different GPU architectures